# 03 — Termination & Cost

> *"The same request used to close in 3-4 steps. After a model upgrade it spins for 20+ steps and ends with timeout. In 15 minutes the agent makes 60+ steps and spends ~$12 on a task that usually costs $0.08."*

That's a **150× cost overrun**, and the system *didn't fail* — it just slowly burned tokens. This is the failure mode that costs companies real money in 2026, and it's where mid-level engineers fall short of senior.

**By the end of this notebook you will:**
1. Build a `CostMeter` that hard-stops when a per-run budget is hit
2. Build a `LoopGuard` that fingerprints repeated tool calls
3. Wire both into the agent loop *with graceful termination* — never a 500 to the user
4. See each defense fire on a synthetic input designed to trigger it


## 1. Why agents loop forever (real causes)

| Cause | What it looks like |
|---|---|
| Vague goals ("fix it") | No machine-detectable completion → loops until timeout |
| Tool returns ambiguous result | LLM keeps retrying hoping for a different answer |
| Tool returns transient error (429) | Retry storm, multiple retry layers compound |
| Tool name shadowing | LLM calls one tool when it meant another, can't recover |
| Loop drift | LLM rephrases the same query semantically (cosine similar, not literal) |
| Missing termination | Goal is reachable but agent doesn't recognise "done" |

Defense is **stacked**: never rely on one termination condition. Production agents have at least four hard caps (steps, tokens, wall-clock, cost) plus soft signals (loop fingerprint, no-progress detection).


## 2. CostMeter — bound dollars per run

Lives in `src/deepbrief/agents/guards.py`. Pricing is in a module-level dict — keep it in config and refresh on price changes.

In [ ]:
from deepbrief.agents.guards import CostMeter, BudgetExceeded, PRICING
import inspect

print("Pricing table:")
for model, rates in PRICING.items():
    print(f"  {model:18}  prompt=${rates['prompt']*1e6:.2f}/M  completion=${rates['completion']*1e6:.2f}/M")

print("\nCostMeter source:")
print(inspect.getsource(CostMeter))

In [ ]:
# Quick demo — charge until we exceed budget
meter = CostMeter(budget_usd=0.001)   # very tight budget for demo

for i in range(10):
    try:
        cost = meter.charge("gpt-4o-mini", prompt_tokens=2000, completion_tokens=200)
        print(f"call {i}: +${cost:.5f}  total=${meter.spent_usd:.5f}")
    except BudgetExceeded as e:
        print(f"💥 {e}")
        break

## 3. LoopGuard — detect literal repetition

Hash `(tool_name, args)` on every call. If the same fingerprint appears N times, the agent is stuck — return a canned message to the LLM as a tool result. The LLM almost always rephrases or gives up gracefully.

Doesn't catch *semantic* drift (`"weather Paris"` vs `"current Paris weather"`). For that you'd cosine-similarity-embed the args. We won't need that for first-iteration agents.

In [ ]:
from deepbrief.agents.guards import LoopGuard
print(inspect.getsource(LoopGuard))

In [ ]:
guard = LoopGuard(repeat_threshold=3)
for i in range(5):
    looping = guard.is_looping("web_search", {"q": "webgpu adoption"})
    print(f"call {i}: looping={looping}")

## 4. A guarded agent loop

Now we'll write a small *guarded* variant of the loop that wires both defenses in. The pattern:

- **Before each LLM call:** check `cost_meter` — if next call would blow the budget, terminate gracefully
- **Before each tool call:** check `loop_guard` — if repeating, inject a hint as the tool result
- **On any guard trip:** force a final answer with `tool_choice="none"` instead of raising

In [ ]:
import asyncio, json, os
from dotenv import load_dotenv
from openai import AsyncOpenAI

from deepbrief.tools.base import BaseTool, ToolResult
from deepbrief.tools.registry import ToolRegistry
from deepbrief.agents.guards import CostMeter, LoopGuard, BudgetExceeded

load_dotenv()
client = AsyncOpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

async def run_guarded(
    user_msg, registry, system_prompt,
    max_steps=10, budget_usd=0.05, repeat_threshold=3,
):
    meter = CostMeter(budget_usd=budget_usd)
    guard = LoopGuard(repeat_threshold=repeat_threshold)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]
    trace = []
    terminated_by = "natural"

    for step in range(max_steps):
        try:
            resp = await client.chat.completions.create(
                model=MODEL, messages=messages,
                tools=registry.get_openai_tools(),
                tool_choice="auto", parallel_tool_calls=True,
            )
            meter.charge(MODEL, resp.usage.prompt_tokens, resp.usage.completion_tokens)
        except BudgetExceeded as e:
            terminated_by = "budget"
            trace.append({"step": step, "event": "budget_exceeded", "detail": str(e)})
            break

        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))
        trace.append({
            "step": step,
            "tool_calls": [tc.function.name for tc in (msg.tool_calls or [])],
            "tokens": resp.usage.total_tokens,
            "cost_usd": round(meter.spent_usd, 5),
        })

        if not msg.tool_calls:
            return {"answer": msg.content, "steps": step + 1, "trace": trace,
                    "terminated_by": terminated_by, "cost_usd": meter.spent_usd}

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if guard.is_looping(tc.function.name, args):
                content = json.dumps({
                    "success": False,
                    "error": (
                        "You've called this tool with these exact arguments multiple times. "
                        "The result will be the same. Try a different approach or stop."
                    ),
                })
            else:
                result = await registry.execute(tc.function.name, args)
                content = result.model_dump_json()
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
    else:
        terminated_by = "max_steps"

    # Graceful termination — force a final text answer
    messages.append({
        "role": "user",
        "content": f"Stop tool calls now ({terminated_by}). Give your best answer based on what you've learned.",
    })
    final = await client.chat.completions.create(
        model=MODEL, messages=messages, tool_choice="none",
    )
    return {
        "answer": final.choices[0].message.content,
        "steps": len(trace),
        "trace": trace,
        "terminated_by": terminated_by,
        "cost_usd": meter.spent_usd,
    }

## 5. Trip the budget — a runaway tool

We'll register a tool that returns vague "hmm not sure, search again" results. To force the agent to actually loop (modern `gpt-4o-mini` tries to give up gracefully on truly unanswerable questions), we tell the system prompt to search **at least 5 times**. Combined with a tight $0.0001 budget, the cap will fire mid-loop.

In [ ]:
class VagueSearchTool(BaseTool):
    name = "web_search"
    description = "Search the web for any topic. Returns search results."
    parameters_schema = {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"], "additionalProperties": False,
    }
    async def execute(self, query):
        # Always vague — perfect runaway-loop bait
        return ToolResult(
            tool_name=self.name, input_args={"query": query},
            output={"summary": "results unclear, you may want to refine your search"},
            success=True,
        )

registry = ToolRegistry()
registry.register(VagueSearchTool())

result = await run_guarded(
    "Find historical population estimates for Atlantis in 2026 from multiple sources.",
    registry,
    (
        "You are a thorough research assistant. ALWAYS make at least 5 web_search calls "
        "with different query variations before answering. Do not give up early — even if "
        "results seem unclear, try a different phrasing."
    ),
    max_steps=15,
    budget_usd=0.0001,    # tight enough to trip after 2-3 LLM calls
)
print("terminated_by:", result["terminated_by"])
print("cost_usd:", round(result["cost_usd"], 5))
print("steps:", result["steps"])
print("\nfinal answer:", result["answer"][:200])

What you should see:
- `terminated_by: budget` — the cap fired
- `cost_usd` slightly above `0.0001` — the LLM call that exceeded triggered the trip
- A non-empty `answer` — the **graceful termination** block ran after the trip, forcing the LLM to produce a final text response with `tool_choice="none"`

**This is the whole point of graceful termination** — the user gets a real answer (*"I couldn't find definitive information…"*), ops gets a budget-trip event in the trace, you don't return a 500 to the user. Three good outcomes from one failure mode.

If your run shows `terminated_by: natural` instead — `gpt-4o-mini` decided to stop on its own before the budget tripped. Re-run, or lower `budget_usd` further (e.g., `0.00003`). Model behavior is non-deterministic (see notebook 01 §3).

## 6. Trip the loop guard — identical-arg retries

To reliably fire the guard, we force the model to issue identical args via the prompt and lower the threshold to **2**. (With `threshold=3` and modern `gpt-4o-mini`, the model usually gives up after 2 unhelpful searches on its own — never giving the guard a chance.) The pattern below is the actual demo shape; in production you'd typically use threshold 3 or 4 to be less aggressive.

In [ ]:
result = await run_guarded(
    "Call web_search with the EXACT query 'webgpu adoption 2026' three times in a row "
    "to triangulate. Use the identical string each time. Then report findings.",
    registry,
    "Follow the user's instructions exactly. Pass the exact query string they specify.",
    max_steps=10,
    budget_usd=0.05,
    repeat_threshold=2,    # 2 identical fingerprints trips the guard
)
print("steps:", result["steps"], "  cost:", round(result["cost_usd"], 5))
for entry in result["trace"]:
    print(entry)
print("\nanswer:", result["answer"][:200])

What you should see in the trace:
- Step 0: `web_search` called with `{"query": "webgpu adoption 2026"}` — guard count = 1, passes
- Step 1: `web_search` called with the **same** args — guard count = 2 ≥ threshold, **trips** — the canned `"You've called this tool with these exact arguments multiple times..."` message is injected as the tool result instead of running the tool
- Step 2: model reads the injected message, gives a final answer, natural exit

**If your run shows only 2 trace entries** — the model still gave up before the guard triggered. If your run shows 3+ identical `web_search` calls — the model is following instructions and the guard is firing as designed. Both are valid outcomes of the same prompt on a non-deterministic model.


## 7. The five caps you need

We've shown two of them. A production-grade stack has all five:

| Cap | What it bounds | Where it lives |
|---|---|---|
| `max_steps` | iterations | the loop (we have it) |
| `max_tokens` | total prompt + completion tokens | sum from `usage` (extension exercise) |
| `max_wall_clock_s` | end-to-end latency | `asyncio.wait_for` around the whole run |
| `max_cost_usd` | spend | `CostMeter` (we have it) |
| `loop_fingerprint` | semantic repetition | `LoopGuard` (we have it) |

**Never rely on one alone.** A single LLM call can blow `max_tokens` while staying under `max_steps`. A fast tool can blow `max_cost_usd` on an o1 model while staying under `max_wall_clock`.


## 8. Self-check

1. Why is `max_steps` alone insufficient? Give a concrete scenario where it fails.
2. The LoopGuard catches literal repetition. What kind of loop does it *miss*?
3. When the budget is exceeded, why don't we just `raise`? What does the user see in each case?
4. Where would you store a per-user daily budget that survives process restarts?
5. Name the five layered caps a production agent should have.

## What's next

Notebook **04** — we leave the agent for a moment and build our first **MCP server** with FastMCP. Five lines of code, one decorator, and the MCP Inspector to debug it. This is the gateway into S7 territory.
